In [ ]:
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


def find_project_root() -> Path:
    try:
        here = Path(__file__).resolve().parent  # type: ignore[name-defined]
    except NameError:
        here = Path.cwd().resolve()

    candidates = [here, *here.parents, Path.cwd().resolve()]
    candidates.extend([
        Path("/content/1.C51-ML-Project"),
        Path("/content/drive/MyDrive/1.C51-ML-Project"),
    ])

    for cand in candidates:
        try:
            cand = cand.expanduser().resolve()
        except Exception:
            continue
        if (cand / "README.md").exists() and (cand / "requirements.txt").exists():
            return cand
    return Path.cwd().resolve()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
CARBON_DIR = DATA_DIR / "carbon"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = PROCESSED_DIR
NESO_PATH = CARBON_DIR / "neso_historic_generation_mix.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 1. Results table ──────────────────────────────────────────────
results_data = {
    "scenario": [
        "Observed baseline",
        "Cold winter −2°C",
        "Severe cold −3°C",
        "Retrofit 10%",
        "Cold + retrofit 10%",
        "Cold + retrofit + ToU flexibility",
        "Strong policy package",
    ],
    "delta_demand_pct": [0.00,  0.21,  0.30, -10.03,  -9.87, -10.60, -15.84],
    "delta_peak_pct":   [0.00,  0.22,  0.33,  -9.96,  -9.79, -16.10, -23.28],
    "total_kgCO2":      [194344.92, 194756.08, 194945.93, 174847.95,
                         175161.16, 173691.46, 163486.83],
    "delta_emissions_pct": [0.00, 0.21, 0.31, -10.03, -9.87, -10.63, -15.88],
}

df_results = pd.DataFrame(results_data)
df_results["kgCO2_saved"] = (
    df_results.loc[df_results["scenario"] == "Observed baseline", "total_kgCO2"].values[0]
    - df_results["total_kgCO2"]
)

# ── 2. Save to CSV ────────────────────────────────────────────────
csv_path = RESULTS_DIR / "scenario_emissions_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"Saved to: {csv_path}")
print(df_results.to_string(index=False))

# ── 3. Plot setup ─────────────────────────────────────────────────
scenarios_plot = df_results[df_results["scenario"] != "Observed baseline"].copy()
labels = [s.replace(" + ", "\n+ ") for s in scenarios_plot["scenario"]]
x = np.arange(len(labels))
w = 0.26

BLUE   = "#378ADD"
GREEN  = "#1D9E75"
CORAL  = "#D85A30"
GRAY   = "#888780"

fig, axes = plt.subplots(2, 1, figsize=(12, 11))
fig.suptitle("Scenario analysis: demand and emissions impact", fontsize=14, fontweight="500", y=0.98)

# ── Chart 1: grouped % change bars ───────────────────────────────
ax = axes[0]
b1 = ax.bar(x - w,   scenarios_plot["delta_demand_pct"],   w, label="Δ total demand (%)",   color=BLUE,  zorder=3)
b2 = ax.bar(x,       scenarios_plot["delta_emissions_pct"], w, label="Δ emissions (%)",       color=GREEN, zorder=3)
b3 = ax.bar(x + w,   scenarios_plot["delta_peak_pct"],      w, label="Δ evening peak (%)",    color=CORAL, zorder=3)

ax.axhline(0, color="black", linewidth=0.8, zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Change vs baseline (%)", fontsize=11)
ax.set_title("Percentage change in demand, emissions, and evening peak", fontsize=11, pad=10)
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.spines[["top", "right"]].set_visible(False)

# value labels on bars
for bars in [b1, b2, b3]:
    for bar in bars:
        v = bar.get_height()
        if abs(v) > 0.05:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                v + (0.3 if v >= 0 else -0.7),
                f"{v:+.1f}%",
                ha="center", va="bottom" if v >= 0 else "top",
                fontsize=7.5, color="black"
            )

legend_patches = [
    mpatches.Patch(color=BLUE,  label="Δ total demand (%)"),
    mpatches.Patch(color=GREEN, label="Δ emissions (%)"),
    mpatches.Patch(color=CORAL, label="Δ evening peak (%)"),
]
ax.legend(handles=legend_patches, fontsize=9, framealpha=0.5)

# ── Chart 2: absolute kgCO2 saved ────────────────────────────────
ax2 = axes[1]
colors = [GREEN if v >= 0 else CORAL for v in scenarios_plot["kgCO2_saved"]]
bars2 = ax2.barh(labels, scenarios_plot["kgCO2_saved"], color=colors, zorder=3)

ax2.axvline(0, color="black", linewidth=0.8, zorder=2)
ax2.set_xlabel("kgCO₂ saved vs baseline", fontsize=11)
ax2.set_title("Absolute emissions reduction by scenario", fontsize=11, pad=10)
ax2.grid(axis="x", alpha=0.3, zorder=0)
ax2.spines[["top", "right"]].set_visible(False)

for bar, v in zip(bars2, scenarios_plot["kgCO2_saved"]):
    offset = 200 if v >= 0 else -200
    ha = "left" if v >= 0 else "right"
    ax2.text(
        v + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{v:,.0f} kgCO₂",
        va="center", ha=ha, fontsize=9
    )

legend_patches2 = [
    mpatches.Patch(color=GREEN, label="Emissions saved"),
    mpatches.Patch(color=CORAL, label="Extra emissions"),
]
ax2.legend(handles=legend_patches2, fontsize=9, framealpha=0.5)

plt.tight_layout()

# ── 4. Save figure ────────────────────────────────────────────────
fig_path = RESULTS_DIR / "scenario_emissions_chart.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to: {fig_path}")